# ⛰ Core Lab 5 — CPU or GPU?
**A Billion Small Steps · Hardware for AI · Step 2 · Book sections 2.5–2.8, 2.C · Foothold lessons 2.5–2.8, 2.C**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

**What you should see:** the GPU winning only for large work; measured speed-up close to Amdahl's prediction; copies erasing the gain.

| Part | What you do | Book |
|---|---|---|
| A | Meet your GPU | 2.5 |
| B | Amdahl's law on a real program | 2.8 |
| C | The big race: matrix multiply, the cost of copying, and the decision tool | 2.C, App. B.2 |

⏱️ About 2 hours. 📘 Textbook: Stallings §2.2–2.3 · H&P §4.4. Course site: https://aishugithub.github.io/course_GPU/

🥾 Side Trails for this Step (clock speed and power, vector add, the decision tool's code) are in the **Step 2 Side Trails** notebook.

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

## Part A · Meet Your GPU  *(Section 2.5)*

PyTorch can tell you about the GPU. A **streaming multiprocessor (SM)** is one of the GPU's "mini-processors"; each one holds many simple cores (Section 3.1).

In [ ]:
# ask PyTorch for the facts about GPU number 0
p = torch.cuda.get_device_properties(0)
print("Name                      :", p.name)   # the GPU's name
# bytes ÷ 1e9 = gigabytes
print("Memory                    :", round(p.total_memory / 1e9, 1), "GB")
print("Streaming multiprocessors :", p.multi_processor_count)   # how many SMs (mini-processors) it has
print("Compute capability        :", f"{p.major}.{p.minor}")   # its design generation (higher = newer)

`nvidia-smi` is the GPU's "task manager". Run it below. Look for the GPU name, memory used / total, and the power and temperature columns.

In [ ]:
# A "!" at the start runs a system command instead of Python. nvidia-smi = NVIDIA's GPU status tool.
!nvidia-smi

### ✏️ YOUR TURN (A)
A T4 has 40 SMs with 64 CUDA cores each. How many CUDA cores is that? Compare with Colab's CPU: run `import os; print(os.cpu_count())` in a new cell.

*Your answer:*

## Part B · Amdahl's Law on a Real Program  *(Section 2.8)*

A pretend "training step": a **serial CPU part** (Python work that no GPU can help with) plus a **matrix multiply** we can move to the GPU. Amdahl predicts the speed-up from the parts; then we time the real new step and compare.

In [ ]:
# two 2048 × 2048 matrices in CPU memory, and copies of them in GPU memory
N = 2048
x_cpu = torch.rand(N, N); w_cpu = torch.rand(N, N)   # two 2048 × 2048 matrices in CPU memory
x_dev = x_cpu.to(device); w_dev = w_cpu.to(device)   # copies of them in GPU memory
# 2 lakh made-up labels for the part of the step that only the CPU can do
labels = list(range(200_000))

def serial_part():   # the part of the step only the CPU can do
    # e.g. cleaning labels in plain Python: one at a time, on the CPU
    return sum(v % 7 for v in labels)

# time the serial part, the multiply on the CPU, and the same multiply on the GPU
t_serial = timeit(serial_part, repeats=3)
t_mm_cpu = timeit(lambda: x_cpu @ w_cpu, repeats=3)   # the multiply on the CPU
t_mm_gpu = timeit(lambda: x_dev @ w_dev, repeats=3)   # the same multiply on the GPU

# the whole step on the CPU; p = the fraction we CAN speed up; s = how much the GPU speeds it up
total_cpu = t_serial + t_mm_cpu
p = t_mm_cpu / total_cpu   # p = the fraction of the time we CAN speed up
s = t_mm_cpu / t_mm_gpu   # s = how many times faster the GPU does that part
# Amdahl's prediction
predicted = 1 / ((1 - p) + p / s)
# the real thing: time the new step (serial part, then the multiply on the GPU) and compare
t_new = timeit(lambda: (serial_part(), x_dev @ w_dev), repeats=3)
measured = total_cpu / t_new   # the real speed-up: old time ÷ new time

print(f"serial part        : {t_serial*1e3:7.1f} ms")   # show every number (× 1e3 turns seconds into ms)
print(f"matmul on CPU      : {t_mm_cpu*1e3:7.1f} ms")   # the multiply on the CPU
print(f"matmul on GPU      : {t_mm_gpu*1e3:7.1f} ms   (s = {s:.0f}×)")   # the multiply on the GPU, and s
print(f"parallel fraction p: {p:.2f}")   # p
print(f"Amdahl predicts    : {predicted:.2f}× overall")   # the prediction
print(f"Measured           : {measured:.2f}× overall")   # the reality
print(f"Ceiling 1/(1-p)    : {1/(1-p):.2f}×  (no GPU can beat this)")   # the best any GPU could ever do here

The prediction and the measurement are rarely exactly equal: timings jitter from run to run, and the GPU can work in the background while the CPU starts its next serial part (Section 4.4).

### ✏️ YOUR TURN (B)
1. Make the serial part 5× bigger (`range(1_000_000)`). What happens to p, the prediction and the ceiling?
2. In one sentence: why can a faster GPU **not** fix a slow data pipeline?

*Your answers:*

## Part C · The Big Race: Matrix Multiply  *(Section 2.C)*

The core operation of AI. Two GPU timings:
- **GPU (data already there)**: the matrices already live in GPU memory.
- **GPU + copies**: copy the inputs from CPU to GPU, multiply, copy the result back (what happens if you're careless).

In [ ]:
# matrix sizes n (each matrix is n × n), and one list of times per contestant
sizes = [64, 128, 256, 512, 1024, 2048, 4096]
res = {"CPU": [], "GPU": [], "GPU + copies": []}   # a dictionary: one list of times per contestant
for n in sizes:   # for each size
    # two matrices in CPU memory, and copies of them in GPU memory
    A = np.random.rand(n, n).astype(np.float32); B = np.random.rand(n, n).astype(np.float32)
    Ag, Bg = torch.from_numpy(A).to(device), torch.from_numpy(B).to(device)   # copies of them in GPU memory
    # the CPU multiply, and the GPU multiply with the data already there
    res["CPU"].append(timeit(lambda: A @ B, repeats=3))
    res["GPU"].append(timeit(lambda: Ag @ Bg, repeats=3))   # the GPU multiply, data already there
    # the careless version: copy A and B to the GPU, multiply, copy the answer back, every time
    res["GPU + copies"].append(timeit(lambda: (torch.from_numpy(A).to(device) @ torch.from_numpy(B).to(device)).cpu(), repeats=3))
    # the GPU's speed in GFLOPS
    gflops = 2 * n**3 / res["GPU"][-1] / 1e9
    print(f"n={n:5d}  CPU {res['CPU'][-1]*1e3:9.2f} ms   GPU {res['GPU'][-1]*1e3:8.2f} ms ({gflops:6.0f} GFLOPS)   GPU+copies {res['GPU + copies'][-1]*1e3:8.2f} ms")   # show all three times for this size
print(f"At n = 4096 the GPU is {res['CPU'][-1] / res['GPU'][-1]:.0f}× faster than the CPU (data already on the GPU).")   # [-1] = the last (biggest) size

In [ ]:
plt.figure(figsize=(6.5, 4))                         # a new chart
for k, m in zip(res, ["o-", "s-", "^--"]):           # one line per contestant, each with its own marker style
    plt.loglog(sizes, res[k], m, label=k)            # log scale on both axes
plt.xlabel("matrix size n"); plt.ylabel("time (s)")  # axis labels
plt.grid(alpha=0.3, which="both"); plt.legend()      # faint grid and colour key
plt.title("Matrix multiply: CPU vs GPU")             # the chart's title
plt.show()                                           # draw it

### Test the capstone's decision tool against reality

In [ ]:
# the capstone's simple model of each machine (the same numbers as in Section 2.C)
CPU_FLOPS, CPU_BW = 0.2e12, 20e9
GPU_FLOPS, GPU_BW = 4e12, 300e9   # GPU: 4 TFLOPS effective, 300 GB/s
LAUNCH, PCIE = 20e-6, 12e9   # starting GPU work costs 20 µs; PCIe copies run at 12 GB/s

def predict(flops, nbytes, data_on_gpu=False):   # predict each machine's time for a job
    # each machine's time = whichever is slower: doing the maths, or moving the bytes
    cpu = max(flops / CPU_FLOPS, nbytes / CPU_BW)
    # the GPU also pays the launch cost, plus the copy over PCIe if the data isn't there yet
    gpu = LAUNCH + (0 if data_on_gpu else nbytes / PCIE) + max(flops / GPU_FLOPS, nbytes / GPU_BW)
    return cpu, gpu   # both predicted times, in seconds

# a 4096 multiply: 2n³ FLOPs; 3 matrices (A, B, C) of n × n numbers × 4 bytes each
n = 4096
pc, pg = predict(2 * n**3, 3 * n * n * 4, data_on_gpu=True)   # the prediction for the 4096 multiply
print(f"4096 matmul  predicted: CPU {pc*1e3:7.1f} ms, GPU {pg*1e3:6.1f} ms")   # show the prediction...
print(f"             measured : CPU {res['CPU'][-1]*1e3:7.1f} ms, GPU {res['GPU'][-1]*1e3:6.1f} ms")   # ...and the measured race result below it

# ReLU on 5 crore numbers three ways: on the CPU; copied over, ReLU, copied back; data already on the GPU
relu_cpu_data = np.random.randn(50_000_000).astype(np.float32)
relu_gpu_data = torch.from_numpy(relu_cpu_data).to(device)   # a copy already in GPU memory
t_cpu = timeit(lambda: np.maximum(relu_cpu_data, 0), repeats=3)   # ReLU on the CPU
t_copy = timeit(lambda: torch.relu(torch.from_numpy(relu_cpu_data).to(device)).cpu(), repeats=3)   # copy over, ReLU on the GPU, copy back
t_gpu = timeit(lambda: torch.relu(relu_gpu_data), repeats=3)   # ReLU on the GPU, data already there
print(f"\nReLU on 5 crore numbers:  CPU {t_cpu*1e3:6.1f} ms | GPU incl. copies {t_copy*1e3:6.1f} ms | GPU, data already there {t_gpu*1e3:6.2f} ms")   # show all three

### ✏️ YOUR TURN (C) — record your results
| | Your value |
|---|---|
| Crossover size (GPU starts winning) | |
| GPU speed-up at n = 4096 | |
| Best GPU GFLOPS measured | |
| ReLU: CPU vs GPU-incl-copies — which won? | |

1. Was the decision tool's prediction in the right ballpark (within about 3×)?
2. Why is "GPU + copies" so much slower than "GPU"? What habit does this teach?

*Your answers:*

---
## 🎓 Wrap-up
- The GPU's cores are slower, but there are thousands of them: throughput beats latency for big, parallel work.
- Tiny jobs, serial code and copy-dominated jobs stay on the CPU.
- Amdahl's law: the serial part sets the ceiling.
- **Next — Core Lab 6: Inside the GPU.** SMs, warps, Tensor Cores and your first CUDA C kernel.